**在多分类神经网络的反向传播**：

对于第 i 层而言：
$$
\delta^i=\delta^{i+1}(w^{i+1})^T \odot act'(z^i)
$$
$$
\frac{\partial loss}{\partial w^i} = (a^{i-1})^T \delta^i
$$
$$
\frac{\partial loss}{\partial b^i} = \delta^i
$$

其中，$\delta^i = a^i - y$是第i层的logits的导数，第一层的输入$a^0=x$

上面是针对一条数据，而在批量数据中，

对于第 i 层而言：
$$
\delta^i = \delta^{i+1}(w^{i+1})^T \odot act'(z^i)
$$
$$
\frac{\partial loss}{\partial w^i} = (a^{i-1})^T \delta^i
$$
$$
\frac{\partial loss}{\partial b^i_j} = \sum^N_{k=1} \delta^i_{kj}
$$

仅偏置的偏导不一样，因为当有batchsize时，截距会被复制增加到相应的行数，对每一个样本都起作用

**梯度消失和梯度爆炸**

以一个3层神经网络的第一层权重为例：

\begin{eqnarray}
\frac{\partial loss}{\partial w^1} &=& X^T \delta^1  \\
&=& X^T \delta^2 (w^2)^T \odot act'(z^1)     \\
&=& X^T \delta^3 (w^3)^T \odot act'(z^2) (w^2)^T \odot act'(z^1) \\
&=& X^T (a^3 - y) (w^3)^T \odot act'(z^2) (w^2)^T \odot act'(z^1)
\end{eqnarray}

可知，每一层权重的梯度值，由后面各层的激活函数的导数值、后面各层的权重值，一起连乘构成。

连乘导致，如果值小于1梯度非常小，参数不更新；如果值大于1梯度很大

**激活函数**

- sigmoid函数：导数在0-0.25之间，容易导致梯度消失，所以只适合作为二分类最后输出层的激活函数
- tanh函数：导数在0-1之间，但如果输入小于-4或大于4时，梯度非常接近0，所以训练时需要确保输入不要偏离0太远
- ReLU函数：只要输入大于0，导数值恒为1，输入小于0，导数值恒为0，更适合用于深度网络

**参数初始化**：
- 随机。每层的不同神经元输入是相同的，需要保证能经过不同的参数后输出不同；
- 控制方差。输入一般都标准化到均值为0方差为1，希望正向传播的时候每层输出的方差不变保持稳定。但每层都是每个输入和权重乘积的累加，有n个输入，方差就变为n。另外，因为采用了ReLU，约一半的输入为0，所以方差为$\frac{n}{2}$。所以，为了保持方差不变，权重初始化为均值为0，标准差为$\sqrt{\frac{2}{n}}$
- 希望方差稳定，是为了输入的信息能够稳定传入深层，信号在传播中不会被放大也不会被压缩，每一层都在有效学习，梯度不会爆炸/消失

In [1]:
import torch
from torch import no_grad
from torch.utils.data import DataLoader, Dataset

手动定义多分类神经网络

In [3]:
class MNISTDataset(Dataset):
    def __init__(self, file_path):
        self.images, self.labels = self._read_file(file_path)

    def _read_file(self, file_path):
        images = []
        labels = []
        with open(file_path, 'r') as f:
            next(f)  # 跳过标题行
            for line in f:
                line = line.rstrip("\n")
                items = line.split(",")
                images.append([float(x) for x in items[1:]])
                labels.append(int(items[0]))
        return images, labels

    def __getitem__(self, index):
        image, label = self.images[index], self.labels[index]
        image = torch.tensor(image)
        image = image / 255.0   # 归一化
        image = (image - 0.1307) / 0.3081   # 标准化，由于图像的信息是通过像素之间的明暗对比产生的，故要对整体进行标准归一化
        label = torch.tensor(label)
        return image, label

    def __len__(self):
        return len(self.images)

# 加载数据
batch_size = 64
train_dataset = MNISTDataset('D:/agent/torch/data/mnist/mnist_train.csv')
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataset = MNISTDataset('D:/agent/torch/data/mnist/mnist_test.csv')
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
learning_rate = 0.1
num_epochs = 10

# 定义神经网络
layer_size = [28*28, 128, 128, 128, 64, 10]

# 手动初始化参数
weights = []
biases = []
for in_size, out_size in zip(layer_size[:-1], layer_size[1:]):
    W = torch.randn(in_size, out_size, device=device) * torch.sqrt(torch.tensor(2/ in_size))    # 乘以sqrt(2/in_size)来初始化权重
    b = torch.zeros(out_size, device=device)
    weights.append(W)
    biases.append(b)

# 激活函数
def relu(x):
    return torch.clamp(x, min=0)

def relu_grad(x):
    return (x > 0).float()

def softmax(x):
    x_exp = torch.exp(x - x.max(dim=1, keepdim=True).values)    # 防止输入的值过大（超过float的表示范围），分子分母的指数部分都减去一个最大值
    return x_exp / x_exp.sum(dim=1, keepdim=True)

# 交叉熵损失函数
def cross_entropy(pred, labels):
    N = pred.shape[0]
    one_hot = torch.zeros_like(pred)
    one_hot[torch.arange(N), labels] = 1    # 生成label的独热编码
    loss = - (one_hot * torch.log(pred + 1e-8)).sum() / N   # 计算平均loss
    return loss, one_hot


## 训练
for epoch in range(num_epochs):
    total_loss = 0
    for images, labels in train_loader:
        x = images.to(device)
        y = labels.to(device)
        N = x.shape[0]

        # 前向传播
        activations = [x]   # 输出的激活值a
        pre_acts = []       # 输出的logits值z
        for W, b in zip(weights[:-1], biases[:-1]):     # 最后一层是单独处理softmax
            z = activations[-1] @ W + b     # z^l = a^(l-1)·W^l + b^l
            pre_acts.append(z)
            a = relu(z)
            activations.append(a)

        # 输出层
        z_out = activations[-1] @ weights[-1] + biases[-1]
        pre_acts.append(z_out)
        y_pred = softmax(z_out)

        # 损失
        loss, one_hot = cross_entropy(y_pred, y)
        total_loss += loss.item()

        # 反向传播
        grads_W = [None] * len(weights)
        grads_b = [None] * len(biases)
        # 输出层梯度
        dL_dz = (y_pred - one_hot) / N      #当loss是交叉熵，输出经过softmax， dL_dz = y_hat - y
        grads_W[-1] = activations[-1].t() @ dL_dz
        grads_b[-1] = dL_dz.sum(dim=0)
        # 隐层梯度
        for i in range(len(weights)-2, -1, -1):     # 反向循环
            dL_dz = dL_dz @ weights[i+1].t() * relu_grad(pre_acts[i])
            grads_W[i] = activations[i].t() @ dL_dz     # activations中只保存了隐层的，故不用i-1
            grads_b[i] = dL_dz.sum(dim=0)

        # 更新参数
        with torch.no_grad():
            for i in range(len(weights)):
                weights[i] -= learning_rate * grads_W[i]
                biases[i] -= learning_rate * grads_b[i]

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

## 测试
with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in test_loader:
        x = images.view(-1, layer_size[0]).to(device)   # view()改变维度，-1表示这个维度未知，自动根据另一个维度进行计算得出
        y = labels.to(device)
        a = x
        # 前向传播 隐层
        for W, b in zip(weights[:-1], biases[:-1]):
            a = relu(a @ W + b)
        # 输出层
        logits = a @ weights[-1] + biases[-1]
        preds = logits.argmax(dim=1)

        correct += preds.eq(y).sum().item()
        total += y.size(0)
    print(f"Test Accuracy: {100 * correct / total:.2f}%")

Epoch 1/10, Loss: 0.2649
Epoch 2/10, Loss: 0.1085
Epoch 3/10, Loss: 0.0757
Epoch 4/10, Loss: 0.0566
Epoch 5/10, Loss: 0.0425
Epoch 6/10, Loss: 0.0336
Epoch 7/10, Loss: 0.0269
Epoch 8/10, Loss: 0.0233
Epoch 9/10, Loss: 0.0183
Epoch 10/10, Loss: 0.0179
Test Accuracy: 97.20%


PyTorch 实现多分类神经网络

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

In [6]:
# 自定义数据集
class MNISTDataset(Dataset):
    def __init__(self, file_path):
        self.images, self.labels = self._read_file(file_path)

    def _read_file(self, file_path):
        images = []
        labels = []
        with open(file_path, 'r') as f:
            next(f)  # 跳过标题行
            for line in f:
                items = line.strip().split(",")
                images.append([float(x) for x in items[1:]])
                labels.append(int(items[0]))
        return images, labels

    def __getitem__(self, index):
        image = torch.tensor(self.images[index], dtype=torch.float32).view(-1)
        image = image / 255.0  # 归一化
        image = (image - 0.1307) / 0.3081  # 标准化
        label = torch.tensor(self.labels[index], dtype=torch.long)
        return image, label

    def __len__(self):
        return len(self.images)


# 定义模型
class NeuralNetwork(nn.Module):         # 继承自nn.Module
    def __init__(self):
        super().__init__()              # 调用父类nn.Module的初始化函数
        self.model = nn.Sequential(     # nn.Sequential顺序链接内部各模块
            nn.Linear(28*28, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, x):               # 直接调用定义的model
        return self.model(x)


# 参数设置
batch_size = 64
learning_rate = 0.1
num_epochs = 10
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 数据加载
train_dataset = MNISTDataset('D:/agent/torch/data/mnist/mnist_train.csv')
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataset = MNISTDataset('D:/agent/torch/data/mnist/mnist_test.csv')
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 模型、损失函数、优化器
model = NeuralNetwork().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

# 训练
model.train()
for epoch in range(num_epochs):
    total_loss = 0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()   # 清空梯度
        loss.backward()         # 反向传播
        optimizer.step()        # 更新参数

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

# 测试
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)
print(f"Test Accuracy: {100 * correct / total:.2f}%")

Epoch 1/10, Loss: 0.4918
Epoch 2/10, Loss: 0.1237
Epoch 3/10, Loss: 0.0852
Epoch 4/10, Loss: 0.0645
Epoch 5/10, Loss: 0.0491
Epoch 6/10, Loss: 0.0400
Epoch 7/10, Loss: 0.0322
Epoch 8/10, Loss: 0.0256
Epoch 9/10, Loss: 0.0208
Epoch 10/10, Loss: 0.0193
Test Accuracy: 96.08%


**神经网络可以拟合任意函数**

理论上，一个隐层和一个输出层的神经网络就可以拟合任意函数。通过线性变换+非线性激活，可以构造出一组可学习的函数基，从而通过组合逼近任意连续函数

**需要深度神经网络**

深层的神经网络可以逐层提取更高级的特征，这种逐层抽象的能力大大提升了神经网络的能力。完成同样的任务，单层的网络需要更多的神经元，而深层网络用更少参数表达更复杂结构